# Kapitel 4 — Mit Wahrscheinlichkeiten argumentieren · Notebook zum Ausfüllen

*Mathe für KI, Schritt für Schritt*, Belk Siri — Begleit-Notebook.

[In Google Colab öffnen](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/de/ch04_a_completer.ipynb) · Begleitseite: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/de.html

Die zu bearbeitenden Zellen sind mit **ZU ERGÄNZEN** gekennzeichnet. Führen Sie die Zellen der Reihe nach aus: Solange eine Antwort fehlt, melden die Prüfungen das ohne Fehler. Die Lösung steht in `ch04_corrige.ipynb` auf derselben Begleitseite.

**Inhalt.** Daten · Die Berechnungen der Lektionen 4.1 bis 4.5 · Aufgaben 4.1 bis 4.6

**Fragen Sie sich vor jeder Berechnung**, in welcher Menge Sie zählen: Spams, markierte Nachrichten oder empfangene Nachrichten.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Farben des Buchs: eine Farbe = eine Rolle
SPAM, LEGITIME, SIGNALE, DENOM = '#d62728', '#b9b6ae', '#1f4e8c', '#ffd54f'
BLEU, ORANGE = '#2a78d6', '#eb6834'   # „kein Spam verpasst“, Kurve der Normalverteilung
from math import comb

## Daten

Die Kennungen D4.1 und D4.2 sind die des Buchs und der CSV-Dateien. Variablen- und Spaltennamen im Code bleiben französisch: `boite_20` und `boite_2` sind die beiden Postfächer, `signalé` = markiert, `non signalé` = nicht markiert, `légitime` = legitim, `durees` = Bearbeitungszeiten, `loi` = Verteilung, `manqué` = verpasst, `moyenne` = Mittelwert, `ecart_type` = Standardabweichung.

In [ ]:
# Daten von Kapitel 4, aus den CSV-Dateien des Buchs übernommen (nicht ändern)
# Zeilen: Spam, legitim. Spalten: markiert, nicht markiert.
LIGNES, COLONNES = ['spam', 'légitime'], ['signalé', 'non signalé']

# D4.1 — Postfach, in dem 20 % der Nachrichten Spam sind (1 000 Nachrichten)
boite_20 = pd.DataFrame([[180, 20],
     [40, 760]],
                        index=LIGNES, columns=COLONNES)

# D4.2 — gleicher Filter, Postfach, in dem 2 % der Nachrichten Spam sind (1 000 Nachrichten)
boite_2 = pd.DataFrame([[18, 2],
     [49, 931]],
                       index=LIGNES, columns=COLONNES)

# D4.3 — 1 000 Bearbeitungszeiten (ms), identisch zur CSV-Datei neu erzeugt
durees = np.random.default_rng(2026).normal(40, 5, 1000).round(1)

## Die Berechnungen der Lektionen in Python

Diese Zellen wiederholen die Berechnungen der Lektionen 4.1 bis 4.5.

In [ ]:
# Lektion 4.1 — zählen, dann durch die richtige Gruppe teilen (Gleichung 4.1)
t = boite_20
n = t.values.sum()
print('gesamt:', n, '| Spams:', t.loc['spam'].sum(), '| markiert:', t['signalé'].sum())
print('P(Spam) =', t.loc['spam'].sum() / n, '| P(markiert) =', t['signalé'].sum() / n,
      '| P(Spam und markiert) =', t.loc['spam', 'signalé'] / n)
# Unter den markierten Nachrichten: Der Nenner ist die Spalte „signalé“
print(f"Spams unter den markierten: {t.loc['spam', 'signalé']} / {t['signalé'].sum()} "
      f"≈ {t.loc['spam', 'signalé'] / t['signalé'].sum():.2f}")

In [ ]:
# Lektion 4.2 — derselbe Filter in einem Postfach mit seltenem Spam (Gleichung 4.2)
d, f = 0.90, 0.05                   # markierte Anteile: der Spams, der legitimen Nachrichten
for p, t in [(0.20, boite_20), (0.02, boite_2)]:
    n_spam, n_leg = round(1000 * p), round(1000 * (1 - p))
    reconstruit = [[round(d * n_spam), n_spam - round(d * n_spam)], [round(f * n_leg), n_leg - round(f * n_leg)]]
    assert (np.array(reconstruit) == t.values).all()   # die Tafel ergibt sich aus den Raten
    print(f'Basisrate {p:.0%}: {t.values.tolist()}  →  Anteil der Spams unter den markierten '
          f'= {d * p / (d * p + f * (1 - p)):.4f}')

In [ ]:
# Lektion 4.3 — zwei bedingte Wahrscheinlichkeiten, zwei Nenner (Gleichung 4.3)
def conditionnelle(t, case, groupe):
    """Anteil des Felds (Zeile, Spalte) in der Gruppe: eine Zeile ('spam') oder eine Spalte ('signalé')."""
    denominateur = t.loc[groupe].sum() if groupe in t.index else t[groupe].sum()
    return t.loc[case] / denominateur, denominateur

for nom, t in [('20 %', boite_20), ('2 %', boite_2)]:
    rappel, den_r = conditionnelle(t, ('spam', 'signalé'), 'spam')
    precision, den_p = conditionnelle(t, ('spam', 'signalé'), 'signalé')
    print(f'Postfach mit {nom}: P(markiert | Spam) = {rappel:.2f} (Nenner {den_r}); '
          f'P(Spam | markiert) = {precision:.2f} (Nenner {den_p})')

# Dieselben Zahlen mit scikit-learn, aus Labels Nachricht für Nachricht (falls installiert)
try:
    from sklearn.metrics import precision_score, recall_score
    vrai = [1] * 180 + [1] * 20 + [0] * 40 + [0] * 760      # 1 = Spam
    predit = [1] * 180 + [0] * 20 + [1] * 40 + [0] * 760    # 1 = markiert
    print(f'scikit-learn: Recall = {recall_score(vrai, predit):.2f}, Präzision = {precision_score(vrai, predit):.2f}')
except ImportError:
    print('scikit-learn nicht installiert: Diese Berechnung ist optional.')

In [ ]:
# Lektion 4.4 — der Satz von Bayes (Gleichungen 4.4 und 4.5) und die Kurve des Buchs
def bayes(p, d=0.90, f=0.05):
    p_signale = d * p + f * (1 - p)                  # Gleichung 4.5
    return d * p / p_signale                        # Gleichung 4.4

for p in (0.20, 0.02):
    print(f'P(Spam) = {p}: P(markiert) = {0.9 * p + 0.05 * (1 - p):.3f}, P(Spam | markiert) = {bayes(p):.4f}')
P = np.linspace(0.001, 0.5, 500)
plt.figure(figsize=(6, 3))
plt.plot(100 * P, bayes(P), color=SIGNALE, linewidth=2)
plt.scatter([2, 20], [bayes(0.02), bayes(0.2)], marker='s', color=SPAM, zorder=3)
plt.xlabel('Basisrate: Anteil der Spams (%)'); plt.ylabel('P(Spam | markiert)'); plt.grid(alpha=0.4)
plt.show()

In [ ]:
# Lektion 4.5 — zehn Spams: Binomialverteilung (Gleichung 4.7) und Gegenereignis (Gleichung 4.8)
n, p = 10, 0.1                        # wir zählen über 10 SPAMS; p = Wahrscheinlichkeit, verpasst zu werden
loi = [comb(n, k) * p ** k * (1 - p) ** (n - k) for k in range(n + 1)]
print('0,9^10 =', round(0.9 ** 10, 4), '| mindestens einer verpasst =', round(1 - 0.9 ** 10, 4))
print('P(X = k), k = 0 bis 4:', [round(v, 3) for v in loi[:5]], '| Summe =', round(sum(loi), 6))
couleurs = [BLEU] + [SPAM] * n
plt.figure(figsize=(6, 3))
plt.bar(range(n + 1), loi, color=couleurs, hatch=[''] + ['//'] * n, edgecolor='white')
plt.xlabel('Anzahl verpasster Spams unter 10 Spams'); plt.ylabel('Wahrscheinlichkeit'); plt.show()

# Ein Blick auf die Normalverteilung: Bearbeitungszeiten (D4.3)
m, s = durees.mean(), durees.std()
print(f'Mittelwert = {m:.2f} ms, Standardabweichung = {s:.2f} ms; innerhalb ±1 Standardabweichung: {np.mean(abs(durees - m) <= s):.1%}, '
      f'innerhalb ±2 Standardabweichungen: {np.mean(abs(durees - m) <= 2 * s):.1%}')

## Aufgabe 4.1 — Ein Diagramm lesen: zwanzig Spams

*Begriffe: Binomialverteilung, Gegenereignis (Lektion 4.5).* Lesen Sie zuerst die Abbildung im Buch.

### a) bis d) Ihre Ablesungen

*Ihre Antworten:* …

In [ ]:
# ZU ERGÄNZEN: die Binomialwahrscheinlichkeit P(X = k) (Gleichung 4.7)
def binomiale(n, p, k):
    return None   # ZU ERGÄNZEN; comb(n, k) liefert den Binomialkoeffizienten

In [ ]:
# Prüfung an den 20 Spams der Aufgabe
if binomiale(20, 0.05, 0) is None:
    print('Ergänzen Sie die Funktion.')
else:
    p0, p1 = binomiale(20, 0.05, 0), binomiale(20, 0.05, 1)
    print(f'P(X = 0) = {p0:.4f}, P(X = 1) = {p1:.4f}, mindestens einer = {1 - p0:.4f} →',
          'richtig' if abs(p0 - 0.95 ** 20) < 1e-9 and abs(p1 - 20 * 0.05 * 0.95 ** 19) < 1e-9 else 'bitte prüfen')

## Aufgabe 4.2 — Ein vorsichtigerer Filter

*Begriffe: bedingte Wahrscheinlichkeit (Lektion 4.3), Satz von Bayes (Lektion 4.4), Basisrate (Lektion 4.2).* Der vorsichtige Filter markiert 75 % der Spams und 1 % der legitimen Nachrichten. Postfach mit 10 000 Nachrichten, davon 10 % Spam. Lösen Sie zuerst die Fragen a) bis e) auf Papier.

### a) bis c) Auf Papier

*Ihre Antworten:* …

### a) Die Tafel in Python

In [ ]:
# ZU ERGÄNZEN: die vier Häufigkeiten der Tafel, für 10 000 Nachrichten
spam_signale = None
spam_non_signale = None
legitime_signale = None
legitime_non_signale = None

In [ ]:
# Prüfung
cases = [spam_signale, spam_non_signale, legitime_signale, legitime_non_signale]
if None in cases:
    print('Ergänzen Sie die vorige Zelle.')
else:
    prudent = pd.DataFrame([cases[:2], cases[2:]], index=LIGNES, columns=COLONNES)
    print(prudent)
    print('Tafel:', 'richtig' if cases == [750, 250, 90, 8910] else 'bitte prüfen')

### b) Die beiden bedingten Wahrscheinlichkeiten

In [ ]:
# ZU ERGÄNZEN: In welcher Gruppe zählen Sie? Schreiben Sie den Quotienten mit den Häufigkeiten.
p_signale_si_spam = None
p_spam_si_signale = None

In [ ]:
# Prüfung
for nom, la_votre, vrai in [('P(markiert | Spam)', p_signale_si_spam, 0.75),
                            ('P(Spam | markiert)', p_spam_si_signale, 750 / 840)]:
    print(f'{nom}:', 'noch nicht ausgefüllt' if la_votre is None else
          ('richtig' if abs(la_votre - vrai) < 0.001 else 'bitte prüfen'))

### c) bis e) Der Satz von Bayes als Funktion

In [ ]:
# ZU ERGÄNZEN: P(Spam | markiert) aus der Basisrate p und den Raten d und f des Filters
def proba_spam_si_signale(p, d, f):
    return None   # ZU ERGÄNZEN (Gleichungen 4.4 und 4.5)

In [ ]:
# Prüfung von c), d) und e)
if proba_spam_si_signale(0.1, 0.75, 0.01) is None:
    print('Ergänzen Sie die Funktion.')
else:
    cas = [('c) vorsichtig, 10 % Spam', (0.10, 0.75, 0.01), 0.8929),
           ('d) vorsichtig, 1 % Spam', (0.01, 0.75, 0.01), 0.4310),
           ('e) vorsichtig, 2 % Spam', (0.02, 0.75, 0.01), 0.6048),
           ('e) Filter der Lektionen, 2 % Spam', (0.02, 0.90, 0.05), 0.2687)]
    for nom, args, attendu in cas:
        v = proba_spam_si_signale(*args)
        print(f'{nom}: {v:.4f} →', 'richtig' if abs(v - attendu) < 0.0001 else 'bitte prüfen')
    P = np.linspace(0.001, 0.5, 500)
    plt.figure(figsize=(6, 3))
    plt.plot(100 * P, proba_spam_si_signale(P, 0.90, 0.05), color=SIGNALE, linewidth=2,
             label='Filter der Lektionen (90 %; 5 %)')
    plt.plot(100 * P, proba_spam_si_signale(P, 0.75, 0.01), color=SPAM, linewidth=2, linestyle='--',
             label='vorsichtiger Filter (75 %; 1 %)')
    plt.axvline(2, color='black', linewidth=0.8, linestyle=':')
    plt.xlabel('Basisrate: Anteil der Spams (%)'); plt.ylabel('P(Spam | markiert)')
    plt.legend(); plt.grid(alpha=0.4); plt.show()

### d) und e) Ihre Antworten

*Ihre Vorhersage zu d), vor dem Rechnen:* …

*Ihre Wahl in e):* …

## Aufgabe 4.3 — Drei Spams, dann zehn Nachrichten

*Begriffe: Bernoulli-Experiment und Binomialverteilung (Lektion 4.5), bedingte Wahrscheinlichkeit (Lektion 4.3).* Lösen Sie die Fragen zuerst auf Papier.

### a) bis e) Auf Papier

*Ihre Antworten:* …

In [ ]:
# ZU ERGÄNZEN: die Wahrscheinlichkeiten für 0, 1, 2, 3 verpasste Spams unter 3, dann die Antwort zu d)
loi_3 = None        # Liste mit vier Zahlen
p_d = None          # mindestens ein Spam kommt durch, unter 10 empfangenen Nachrichten

In [ ]:
# Prüfung
if loi_3 is None or p_d is None:
    print('Ergänzen Sie die vorige Zelle.')
else:
    print('b):', [round(v, 3) for v in loi_3], '→', 'richtig' if np.allclose(loi_3, [0.729, 0.243, 0.027, 0.001]) else 'bitte prüfen')
    print(f'd): {p_d:.4f} →', 'richtig' if abs(p_d - 0.1829) < 0.0001 else 'bitte prüfen')

## Aufgabe 4.4 — Zehn Spams simulieren

*Begriffe: Binomialverteilung (Lektion 4.5), Häufigkeit und Wahrscheinlichkeit.* Wir zählen über **zehn Spams**.

In [ ]:
# ZU ERGÄNZEN: eine Serie von 10 Spams, jeder mit Wahrscheinlichkeit 0,1 verpasst
def simuler_serie(rng):
    """Gibt die Anzahl verpasster Spams unter 10 zurück. rng.random() zieht eine Zahl zwischen 0 und 1."""
    return None   # ZU ERGÄNZEN

In [ ]:
# b) und c) 10 000 Serien (fester Seed, um dieselben Zahlen zu erhalten)
rng = np.random.default_rng(4)
if simuler_serie(rng) is None:
    print('Ergänzen Sie die Funktion.')
else:
    rng = np.random.default_rng(4)
    X = np.array([simuler_serie(rng) for _ in range(10000)])
    freq = np.mean(X >= 1)
    print(f'beobachtete Häufigkeit: {freq:.4f}; berechnete Wahrscheinlichkeit: {1 - 0.9 ** 10:.4f} →',
          'richtig' if abs(freq - (1 - 0.9 ** 10)) < 0.02 else 'bitte prüfen')
    k = np.arange(6)
    plt.figure(figsize=(6, 3))
    plt.bar(k - 0.2, [np.mean(X == v) for v in k], width=0.4, color=SIGNALE, label='beobachtete Häufigkeit')
    plt.bar(k + 0.2, [comb(10, v) * 0.1 ** v * 0.9 ** (10 - v) for v in k], width=0.4, color=SPAM,
            hatch='//', edgecolor='white', label='Wahrscheinlichkeit (Gleichung 4.7)')
    plt.xlabel('Verpasste Spams unter 10'); plt.legend(); plt.show()
    # d) Mehr Serien: Die Häufigkeit nähert sich der Wahrscheinlichkeit
    for N in (100, 100, 100, 100000):
        Y = np.array([simuler_serie(rng) for _ in range(N)])
        print(f'{N:>6} Serien: Häufigkeit = {np.mean(Y >= 1):.4f}')

### d) Ihre Deutung

*Ihre Antwort:* …

## Aufgabe 4.5 — Vorhersagen, dann prüfen: eine Spam-Kampagne

*Begriffe: Unabhängigkeit, Gegenereignis (Lektion 4.5).*

### a) bis c) Ihre Vorhersagen und Berechnungen

*Ihre Vorhersage:* …

In [ ]:
# ZU ERGÄNZEN: eine Kampagne von 10 identischen Spams, alle zusammen mit Wahrscheinlichkeit 0,1 verpasst
def simuler_campagne(rng):
    return None   # ZU ERGÄNZEN: gibt 10 oder 0 zurück

In [ ]:
# d) Prüfung durch Simulation
rng = np.random.default_rng(5)
if simuler_campagne(rng) is None:
    print('Ergänzen Sie die Funktion.')
else:
    rng = np.random.default_rng(5)
    Z = np.array([simuler_campagne(rng) for _ in range(10000)])
    print(f'mindestens einer verpasst: {np.mean(Z >= 1):.4f}; Mittelwert: {Z.mean():.3f} →',
          'richtig' if abs(np.mean(Z >= 1) - 0.1) < 0.01 and abs(Z.mean() - 1) < 0.1 else 'bitte prüfen')

## Aufgabe 4.6 — Herausforderung: Ab wie vielen Spams liegt das Risiko, einen zu verpassen, über 99 %?

*Begriffe: Bernoulli- und Binomialverteilung, Normalverteilung (Lektion 4.5), Standardabweichung (Lektion 1.1).*

### a) bis d) Ihre Beweise und Berechnungen

*Ihre Antworten:* …

In [ ]:
# ZU ERGÄNZEN: b) das kleinste n mit 0,9 ** n < 0,01; c) Mittelwert und Standardabweichung für n = 100
n_min = None
moyenne_100 = None
ecart_type_100 = None

In [ ]:
# Prüfung, dann d) Binomialverteilung gegen Normalverteilung
if None in (n_min, moyenne_100, ecart_type_100):
    print('Ergänzen Sie die vorige Zelle.')
else:
    print(f'n_min = {n_min} →', 'richtig' if n_min == 44 else 'bitte prüfen')
    print(f'Mittelwert = {moyenne_100:.1f}, Standardabweichung = {ecart_type_100:.1f} →',
          'richtig' if np.isclose(moyenne_100, 10) and np.isclose(ecart_type_100, 3) else 'bitte prüfen')
    k = np.arange(0, 26)
    plt.figure(figsize=(6, 3))
    plt.bar(k, [comb(100, v) * 0.1 ** v * 0.9 ** (100 - v) for v in k], color=SPAM, alpha=0.8,
            label='Binomialverteilung (n = 100, p = 0,1)')
    xs = np.linspace(0, 25, 300)
    plt.plot(xs, np.exp(-(xs - 10) ** 2 / 18) / (3 * np.sqrt(2 * np.pi)), color=ORANGE, linewidth=2,
             label='Normalverteilung (Mittelwert 10, Standardabweichung 3)')
    plt.xlabel('Verpasste Spams unter 100'); plt.legend(); plt.show()